In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from cxd.io.frames import load_mat

# Synthetic scan: run `python examples/synthetic_scan.py` first (no measured data).
data_dir: Path = Path("../examples/data")
data_file: Path = data_dir / "scan_0001.mat"

raw_imgs: np.ndarray = load_mat(data_file)

print(f"Images shape: {raw_imgs.shape}")
threshold = 2.5
imgs = raw_imgs > threshold
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(np.log1p(imgs.sum(axis=0)), cmap="viridis")

In [ ]:
from cxd.mapping.spherical import map2spherical
from scipy.ndimage import center_of_mass

com = np.array(center_of_mass(imgs))

_, hist, u_edges, phi_edges = map2spherical(imgs, com, bins=200)
hist_threshold = 1.5
hist = np.where(hist > hist_threshold, hist, 0)
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(
    np.log1p(hist.T),
    extent=(u_edges[0], u_edges[-1], phi_edges[0], phi_edges[-1]),
    aspect="auto",
    cmap="viridis",
)
ax.set_xlabel("u (cos(theta))")
ax.set_ylabel("phi (radians)")
ax.set_title("2D Solid Angle Map")

cbar = plt.colorbar(ax.images[0], ax=ax)
cbar.set_label("log1p(Counts)")

In [ ]:
def unit2spherical(coords: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    p_u = coords[:, 0]
    p_phi = np.arctan2(coords[:, 1], coords[:, 2])
    return p_u, p_phi

In [ ]:
from scipy.ndimage import gaussian_filter

from cxd.mapping.spherical import extract_peak_from_2d_map, map2spherical, unit2spherical

hist_threshold = 1.5

u_vecs, hist, u_edges, phi_edges = map2spherical(imgs, com, bins=150)
hist = np.where(hist > hist_threshold, hist, 0)
hist = gaussian_filter(hist, sigma=1)
# 3. 피크 추출 (DBSCAN)
# eps=0.1은 약 5.7도 이내의 점들을 하나로 묶음
peaks = extract_peak_from_2d_map(hist, u_edges, phi_edges, eps=0.2, threshold_rel=0.3)

print(f"검출된 피크 개수: {len(peaks)}")

# 4. 시각화
fig, ax = plt.subplots(figsize=(8, 6))
# 히스토그램 그리기
im = ax.imshow(
    np.log1p(hist.T),
    extent=[u_edges[0], u_edges[-1], phi_edges[0], phi_edges[-1]],
    aspect="auto",
    origin="lower",
    cmap="viridis",
)

# 추출된 피크를 u, phi로 변환하여 맵 위에 표시
if len(peaks) > 0:
    p_u, p_phi = unit2spherical(peaks)
    ax.scatter(p_u, p_phi, color="black", marker="x", s=100, label="Extracted Peaks")

ax.set_xlabel("cos(theta)")
ax.set_ylabel("phi (rad)")
ax.legend()
# colorbar 추가
cbar = plt.colorbar(im, ax=ax)
plt.show()


print(f"Peaks: {peaks}")

In [ ]:
from scipy.ndimage import map_coordinates


def get_thick_line_profile(
    imgs: np.ndarray,
    com: np.ndarray,
    direction: np.ndarray,
    length: int = 100,
    thickness: int = 2,
) -> np.ndarray:
    """
    특정 방향으로 두께를 가진 라인 프로파일을 추출합니다.
    """
    # 1. 선을 따라갈 샘플링 포인트들 (t: 0 ~ length)
    t = np.arange(length)

    # 2. 두께를 만들기 위한 오프셋 생성 (예: 3x3x3 영역 혹은 주변 픽셀)
    # 여기서는 간단하게 중심 주변의 정수 오프셋 범위를 생성합니다.
    offsets = np.arange(-thickness, thickness + 1)
    oz, oy, ox = np.meshgrid(offsets, offsets, offsets, indexing="ij")
    offsets_vec = np.stack([oz.ravel(), oy.ravel(), ox.ravel()], axis=1)  # (N_offsets, 3)

    # 3. 모든 샘플링 포인트에 대해 오프셋 적용
    # 결과 구조: (N_offsets, length, 3)
    # P(t) = CoM + t*n + Offset
    line_points = com + t[:, None] * direction  # (length, 3)
    all_points = line_points[None, :, :] + offsets_vec[:, None, :]  # (N_offsets, length, 3)

    # 4. map_coordinates를 위해 차원 재조정 (3, N_total_points)
    flat_points = all_points.reshape(-1, 3).T

    # 5. 3D 볼륨에서 값 추출 (order=1: 선형 보간)
    values = map_coordinates(imgs, flat_points, order=1, mode="constant", cval=0.0)

    # 6. 오프셋별로 다시 나누고 평균 계산
    values = values.reshape(len(offsets_vec), length)
    return np.mean(values, axis=0)

In [ ]:
def get_symmetric_profiles(imgs, com, peak_vectors, length=50, thickness=2):
    """
    서로 반대 방향인 피크끼리 짝을 지어 3개의 통합 프로파일을 생성합니다.
    """
    num_peaks = len(peak_vectors)
    used = [False] * num_peaks
    combined_results = []

    for i in range(num_peaks):
        if used[i]:
            continue

        # 현재 피크 i와 가장 '반대'인 피크 j 찾기 (내적이 -1에 가장 가까운 것)
        best_j = -1
        min_dot = 0  # 0보다 작아야 반대 방향

        for j in range(num_peaks):
            if i == j or used[j]:
                continue
            dot = np.dot(peak_vectors[i], peak_vectors[j])
            if dot < min_dot:
                min_dot = dot
                best_j = j

        if best_j != -1:
            used[i] = used[best_j] = True

            # i 방향(+축)과 best_j 방향(-축) 프로파일 추출
            pos_profile = get_thick_line_profile(imgs, com, peak_vectors[i], length, thickness)
            neg_profile = get_thick_line_profile(imgs, com, peak_vectors[best_j], length, thickness)

            # neg_profile을 뒤집어서 하나의 축으로 합침
            # [-length, ..., 0, ..., +length]
            full_profile = np.concatenate([neg_profile[::-1], pos_profile[1:]])
            dist_axis = np.linspace(-length, length, len(full_profile))

            combined_results.append(
                {
                    "pair": (i, best_j),
                    "dist": dist_axis,
                    "profile": full_profile,
                    "dot": min_dot,
                }
            )

    return combined_results


def plot_paired_lines_subplots(combined_results):
    num_axes = len(combined_results)
    # 3개의 서브플롯 생성 (세로로 배치)
    # sharex=True: 모든 그래프의 X축 범위를 통일하여 비교를 용이하게 함
    fig, axes = plt.subplots(num_axes, 1, figsize=(10, 4 * num_axes), sharex=True)

    # 만약 결과가 1개뿐일 경우 axes가 배열이 아니게 되는 상황 방지
    if num_axes == 1:
        axes = [axes]

    colors = plt.cm.plasma(np.linspace(0.2, 0.8, num_axes))

    for i, (ax, res) in enumerate(zip(axes, combined_results)):
        p1, p2 = res["pair"]

        # 메인 프로파일 플롯
        ax.plot(
            res["dist"],
            res["profile"],
            color=colors[i],
            lw=2.5,
            label=f"Axis {i + 1}: Peaks {p1 + 1} & {p2 + 1}",
        )

        # 중심선(CoM) 표시
        ax.axvline(0, color="white", linestyle="--", alpha=0.3)

        # 서브플롯별 설정
        ax.set_title(f"Reciprocal Axis {i + 1} (Dot Product: {res['dot']:.4f})", fontsize=12)
        ax.set_ylabel("Intensity")
        ax.grid(True, which="both", alpha=0.1)
        ax.legend(loc="upper right")

    # 최하단 그래프에만 X축 라벨 표시
    axes[-1].set_xlabel("Distance along axis (pixels)")

    plt.tight_layout()
    plt.show()


results = get_symmetric_profiles(imgs, com, peaks, length=50, thickness=3)
plot_paired_lines_subplots(results)

In [ ]:
import matplotlib.pyplot as plt


def plot_peak_profiles(imgs, com, peak_vectors, length=None, thickness=5):
    if length is None:
        length = int(np.min(imgs.shape) / 2)  # 볼륨 절반 크기까지

    plt.figure(figsize=(10, 6))

    colors = plt.cm.viridis(np.linspace(0, 1, len(peak_vectors)))

    for i, n_hat in enumerate(peak_vectors):
        # n_hat 방향으로 프로파일 추출
        profile = get_thick_line_profile(imgs, com, n_hat, length, thickness)

        # 거리 축 (pixel)
        dist = np.arange(len(profile))

        plt.plot(dist, profile, label=f"Peak {i + 1}", color=colors[i], lw=2)

    plt.title(f"Intensity Profiles along Peak Directions (Thickness: {thickness}px)")
    plt.xlabel("Distance from CoM (pixels)")
    plt.ylabel("Intensity")
    # plt.yscale('log') # BCDI 데이터는 보통 로그 스케일로 봅니다
    plt.grid(True, which="both", ls="-", alpha=0.5)
    plt.legend()
    plt.show()


# 사용 예시:
plot_peak_profiles(imgs, com, peaks, length=50, thickness=5)

In [ ]:
import numpy as np
import vtk
from vtk.util import numpy_support


def vtk_visualize_cxd(imgs, com, combined_results, thickness=1.0):
    """
    VTK를 사용하여 3D 패턴과 3개의 대칭축을 시각화합니다.
    """
    # --- 1. Renderer 및 RenderWindow 설정 ---
    renderer = vtk.vtkRenderer()
    render_win = vtk.vtkRenderWindow()
    render_win.AddRenderer(renderer)
    render_win.SetSize(1000, 1000)

    interactor = vtk.vtkRenderWindowInteractor()
    interactor.SetRenderWindow(render_win)

    # --- 2. 3D 볼륨 렌더링 (Intensity Pattern) ---
    # NumPy 배열을 VTK 데이터로 변환 (주의: 축 순서 변경)
    # VTK는 (X, Y, Z) 순서, NumPy는 (Z, Y, X)이므로 transpose 필요 없음 (VTK가 알아서 함)
    vtk_data = numpy_support.numpy_to_vtk(
        num_array=imgs.ravel(), deep=True, array_type=vtk.VTK_FLOAT
    )

    volume_data = vtk.vtkImageData()
    volume_data.SetDimensions(imgs.shape[2], imgs.shape[1], imgs.shape[0])  # X, Y, Z
    volume_data.GetPointData().SetScalars(vtk_data)

    # 볼륨 속성 (Color, Opacity)
    volume_property = vtk.vtkVolumeProperty()
    volume_property.ShadeOn()
    volume_property.SetInterpolationTypeToLinear()

    # Opacity: 낮은 강도는 투명하게, 높은 강도는 불투명하게
    opacity_func = vtk.vtkPiecewiseFunction()
    max_val = np.max(imgs)
    opacity_func.AddPoint(0, 0.0)
    opacity_func.AddPoint(max_val * 0.05, 0.02)  # 백그라운드 투명화
    opacity_func.AddPoint(max_val * 0.5, 0.5)
    opacity_func.AddPoint(max_val, 0.8)
    volume_property.SetScalarOpacity(opacity_func)

    # Color: Magma 느낌의 컬러맵
    color_func = vtk.vtkColorTransferFunction()
    color_func.AddRGBPoint(0, 0.0, 0.0, 0.0)
    color_func.AddRGBPoint(max_val * 0.3, 0.5, 0.0, 0.5)
    color_func.AddRGBPoint(max_val, 1.0, 0.8, 0.0)
    volume_property.SetColor(color_func)

    volume_mapper = vtk.vtkGPUVolumeRayCastMapper()
    volume_mapper.SetInputData(volume_data)

    volume_actor = vtk.vtkVolume()
    volume_actor.SetMapper(volume_mapper)
    volume_actor.SetProperty(volume_property)
    renderer.AddActor(volume_actor)

    # --- 3. 3개의 대칭축 그기 (Line Axes) ---
    # CoM 좌표 (VTK는 X, Y, Z 순서이므로 역순)
    com_vtk = [com[2], com[1], com[0]]

    colors = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]  # R, G, B 축

    for i, res in enumerate(combined_results):
        # res['pair']의 첫 번째 피크 벡터 방향 사용
        # combined_results에서 추출된 peak_vectors[p1] 방향 사용
        p1_idx = res["pair"][0]
        direction = peaks[p1_idx]  # [nz, ny, nx]
        dir_vtk = np.array([direction[2], direction[1], direction[0]])

        # 선의 시작점과 끝점 계산 (CoM 중심)
        length = 100  # 선의 길이
        p1 = com_vtk + dir_vtk * length
        p2 = com_vtk - dir_vtk * length

        # VTK Line 생성
        line_source = vtk.vtkLineSource()
        line_source.SetPoint1(p1)
        line_source.SetPoint2(p2)

        line_mapper = vtk.vtkPolyDataMapper()
        line_mapper.SetInputConnection(line_source.GetOutputPort())

        line_actor = vtk.vtkActor()
        line_actor.SetMapper(line_mapper)

        line_actor.GetProperty().SetLineWidth(5)  # 선 두께

        renderer.AddActor(line_actor)

    # --- 4. 기타 설정 (배경, 카메라) ---
    renderer.SetBackground(0.1, 0.1, 0.1)  # 어두운 배경
    renderer.ResetCamera()

    render_win.Render()
    print("VTK 시각화 시작...")
    interactor.Start()


# 사용법:
vtk_visualize_cxd(imgs, com, results)